In [1]:
import napari
import scipy as sp
import numpy as np
import tifffile
import scipy.ndimage as ndi
from skimage.feature import peak_local_max
import os
import glob
import cv2
import skimage as ski
import dask
import plotly.express as px
import pandas as pd
import plotly.graph_objs as go


This is similar to the analysis done for Sid/Ayantika's paper, however the thresholding algorithm used to find the two FISH signals worked quite poorly (they were in a hurry so I did not have much choice), and was regularly throwing away lots of images.  Sid decided to just go ahead and draw ROIs around what he thought was the bright and dim areas, which I will then use to figure out the closest CenpA pair as before.

# Setup Notebook

In [2]:
viewer = napari.Viewer()

In [3]:
# Choose the channel to find CenpA peaks in
peak_channel = 1

In [4]:
scale = [0.10, 0.031, 0.031]

# Utility Functions

In [5]:
def backsub(inp, radius=20):
    filterSize =(radius, radius)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE,
                                    filterSize)
    blurred = cv2.GaussianBlur(inp, (5, 5), 0)
    tophat_img = cv2.morphologyEx(blurred,
                                cv2.MORPH_TOPHAT,
                                kernel)
    rtn = inp.astype(np.single) - (blurred-tophat_img)
    rtn = np.clip(rtn, 0, np.inf)
    return rtn
def backsub_3d(inp, radius=20):
    shape = inp.shape
    reshaped = inp.reshape(-1, shape[-2], shape[-1])
    process = [dask.delayed(backsub)(i, radius) for i in reshaped]
    rslt = np.array(dask.compute(*process))
    rslt = rslt.reshape(shape)
    return rslt

In [6]:
def filter_label_size(labels, min_size=10, max_size=100):
    from skimage import measure
    label_sizes = np.bincount(labels.ravel())
    too_small = label_sizes < min_size
    too_large = label_sizes > max_size
    labels[np.isin(labels, np.where(too_small | too_large))] = 0
    labels = ski.segmentation.relabel_sequential(labels)[0]
    return labels

In [7]:
def find_peaks(img, threshold=0.1, display=False):
    #LoG = -ndi.gaussian_laplace(img/1000, sigma=[2,4,4])
    #LoG = -ndi.gaussian_laplace(img/1000, sigma=[1,2,2])
    
    img = ski.transform.rescale(img, [0.5,1,1], order=1, preserve_range=True)
    LoG = -ndi.gaussian_laplace(img/1000, sigma=[2,2,2])
    max_peaks = peak_local_max(LoG, min_distance=1, threshold_rel=threshold)
    if display:
        viewer.add_image(LoG, blending='additive', colormap='magenta', scale=scale)
        viewer.add_points(max_peaks, n_dimensional=True, size=6, scale=scale, name='AllPeaks')
    return max_peaks

def pair_finder(peaks, max_dist=6, scale=[0.12, 0.04, 0.04]):
    scaled_peaks = peaks.copy().astype(float)
    scaled_peaks[:,0] = scaled_peaks[:,0] * scale[0]
    scaled_peaks[:,1] = scaled_peaks[:,1] * scale[1]
    scaled_peaks[:,2] = scaled_peaks[:,2] * scale[2]
    
    d_matrix = sp.spatial.distance.squareform(sp.spatial.distance.pdist(scaled_peaks))
    d_matrix[d_matrix==0] = 10000
    
    min_pos = np.argmin(d_matrix, axis=0)
    distances = np.min(d_matrix, axis=0)
    sorte = np.argsort(distances)
    
    return_distances = []
    lst = []
    for a in np.arange(0,len(sorte)):
        if distances[sorte[a]]<max_dist:
            if any([all (peaks[sorte[a]]==b) for b in lst]) | any([all(peaks[min_pos][sorte[a]]==b) for b in lst]):
                lst
            elif np.abs(peaks[sorte[a]][0]-peaks[min_pos][sorte[a]][0])>6:
                lst
            else:
                lst.append(peaks[sorte[a]])
                lst.append(peaks[min_pos][sorte[a]])
                return_distances.append(distances[sorte[a]])
    return lst, return_distances

def find_peaks_in_file(fname, display=False, threshold=0.16, viewer=None):
    img = tifffile.imread(fname)
    print(img.shape)
    peaks = find_peaks(img[:,peak_channel,:,:], threshold=threshold, display=display)
    filtered_peaks, distances = pair_finder(peaks, max_dist=1.2, scale=scale)
    
    if display:
        viewer.add_image(img, channel_axis=1, scale=scale)
        viewer.add_points(filtered_peaks[::2], n_dimensional=True, size=6, scale=scale, name='FilteredPeaks', face_color='magenta')
        viewer.add_points(filtered_peaks[1::2], n_dimensional=True, size=6, scale=scale, name='FilteredPeaks', face_color='yellow')
    return filtered_peaks, distances

In [8]:
def get_intensity_df(int_img, labels):
    df = pd.DataFrame(ski.measure.regionprops_table(labels, intensity_image=int_img, properties=['label', 'mean_intensity', 'area']))
    df['total_intensity'] = df['mean_intensity'] * df['area']
    return df.drop(columns=['area'])

def get_props_df(labels):
    df = pd.DataFrame(ski.measure.regionprops_table(labels, properties=['label', 'centroid', 'area']))
    return df

def get_all_props(int_img, labels):
    props_df = get_props_df(labels)
    c0_df = get_intensity_df(int_img[0], labels)
    c1_df = get_intensity_df(int_img[1], labels)
    c2_df = get_intensity_df(int_img[2], labels)
    c3_df = get_intensity_df(int_img[3], labels)
    all_df = props_df.merge(c0_df, on='label', suffixes=('', '_c0'))
    all_df = all_df.merge(c1_df, on='label', suffixes=('', '_c1'))
    all_df = all_df.merge(c2_df, on='label', suffixes=('', '_c2'))
    all_df = all_df.merge(c3_df, on='label', suffixes=('', '_c3'))
    return all_df


In [9]:
from roifile import ImagejRoi

def get_roi_mask(raw_fname):
    tail = os.path.split(raw_fname)[-1]
    fname = raw_fname.replace(tail, 'SUM_' + tail + 'f')
    if os.path.exists(fname)==False:
        fname = fname[0:-1]
        roi_file = fname.replace('.tif', '_ROI.zip')
    else:
        roi_file = fname.replace('.tiff', '_ROI.zip')

    img = ski.io.imread(fname)
    
    if os.path.exists(roi_file)==False:
        return img * 0
    rois = ImagejRoi.fromfile(roi_file)
    img_shape = img.shape[0:2]

    mask = np.zeros(img_shape)
    for idx, roi in enumerate(rois):
        init = np.array([roi.top, roi.left])
        if roi.multi_coordinates is not None:
            print('Bad')
            return mask
        pts = roi.integer_coordinates[:,::-1] + init[np.newaxis, :]
        mask = cv2.fillPoly(mask, [pts[:,::-1]], 2-idx) # He did dim first, I think the rest of the code assumes bright was first
    return mask.astype(int)


In [17]:
def get_mask(peak, img_shape, radius=8):
    blank_img = np.zeros(img_shape)
    cv2.line(blank_img, tuple(peak[0].astype(int)[::-1]), tuple(peak[1].astype(int)[::-1]), color=255, thickness=2)
    edt = ndi.distance_transform_edt(blank_img==0)
    binary = edt < radius
    return binary

def process_file(fname, viewer=None, display=False, max_distance=20):

    # Get peak pairs and distances
    peaks, distances = find_peaks_in_file(fname, display=display, viewer=viewer, threshold=0.16)
    pks_grouped = np.reshape(peaks, (-1, 2, 3))
    centers = np.mean(pks_grouped, axis=1)[:,1:3]
    if len(centers) < 4:
        print(f"Not enough peaks found in {fname}. Skipping file.")
        return pd.DataFrame()

    img = ski.io.imread(fname)
    img = np.moveaxis(img, -1, 0)
    img = backsub_3d(img, radius=20)

    # Get the 2 objects Sid selected
    fimg = img[0,:,:,:]
    pimg = fimg.sum(axis=0)[:,:]
    labels = get_roi_mask(fname)
    if np.max(labels) < 2:
        print(f"Not enough ROIs found in {fname}. Skipping file.")
        return pd.DataFrame()

    df = pd.DataFrame(ski.measure.regionprops_table(labels, intensity_image=pimg, properties=['label', 'area', 'centroid', 'mean_intensity']))
    df['total_intensity'] = df['mean_intensity'] * df['area']
    # Sort or trust him?
    df = df.rename(columns={'centroid-0': 'y', 'centroid-1': 'x'})
    #df = df.sort_values(by='total_intensity', ascending=False).rename(columns={'centroid-0': 'y', 'centroid-1': 'x'})

    # Get the bright and dim centroids from the FISH channel, calculate distances to all pairs' centers
    bright = df.iloc[0][['y','x']].values
    dim = df.iloc[1][['y','x']].values
    bright_distances = np.linalg.norm(centers - bright, axis=1)
    dim_distances = np.linalg.norm(centers - dim, axis=1)

    # Find the closest peak to the bright and dim FISH spots and their distances
    bright_peak = pks_grouped[np.argmin(bright_distances)][:,1:3]
    dim_peak = pks_grouped[np.argmin(dim_distances)][:,1:3]
    bright_peak_distance = bright_distances[np.argmin(bright_distances)]
    dim_peak_distance = dim_distances[np.argmin(dim_distances)]
    if bright_peak_distance > max_distance or dim_peak_distance > max_distance:
        print(f"Bright peak distance {bright_peak_distance} or dim peak distance {dim_peak_distance} exceeds max distance {max_distance}.")
        return pd.DataFrame()

    # Draw line connecting the 2 bright peaks and make a mask, and another for the dim peaks
    bright_mask = get_mask([bright_peak[0], bright_peak[1]], pimg.shape)
    dim_mask = get_mask([dim_peak[0], dim_peak[1]], pimg.shape)
    both_labels = bright_mask + dim_mask * 2

    # Get the intensity information for all channels
    proj_img = img.sum(axis=1)
    prop_df = get_all_props(proj_img, both_labels)

    if display:
        viewer.add_image(proj_img, scale=scale[1:], channel_axis=0)
        viewer.add_labels(labels, name='FISH Labels', scale=scale[1:])
        viewer.add_labels(both_labels, name='Areas', scale=scale[1:])
        viewer.layers[-1].contour = 2
    
    prop_df['FISH_area'] = df['area'].iloc[0:2].values
    prop_df['FISH_total_intensity'] = df['total_intensity'].iloc[0:2].values
    prop_df['FISH_mean_intensity'] = df['mean_intensity'].iloc[0:2].values
    prop_df['distance'] = np.array([bright_peak_distance, dim_peak_distance])
    prop_df['slices'] = img.shape[1]
    
    return prop_df


# Process Data

In [18]:
#fnames = glob.glob('*INCENP*/crop_rep*/*.tif')
fnames = glob.glob('CEN7/*/crop*/*.tif')
fname = fnames[0]
fnames

['CEN7\\Cen7_AuroraB_CENPA\\crop_rep1\\rep1_image001-1.tif',
 'CEN7\\Cen7_AuroraB_CENPA\\crop_rep1\\rep1_image003-1.tif',
 'CEN7\\Cen7_AuroraB_CENPA\\crop_rep1\\rep1_image003-2.tif',
 'CEN7\\Cen7_AuroraB_CENPA\\crop_rep1\\rep1_image004-1.tif',
 'CEN7\\Cen7_AuroraB_CENPA\\crop_rep1\\rep1_image005-1.tif',
 'CEN7\\Cen7_AuroraB_CENPA\\crop_rep1\\rep1_image005-2.tif',
 'CEN7\\Cen7_AuroraB_CENPA\\crop_rep1\\rep1_image005-3.tif',
 'CEN7\\Cen7_AuroraB_CENPA\\crop_rep1\\rep1_image006-1.tif',
 'CEN7\\Cen7_AuroraB_CENPA\\crop_rep1\\rep1_image006-2.tif',
 'CEN7\\Cen7_AuroraB_CENPA\\crop_rep1\\rep1_image007-1.tif',
 'CEN7\\Cen7_AuroraB_CENPA\\crop_rep1\\rep1_image008-1.tif',
 'CEN7\\Cen7_AuroraB_CENPA\\crop_rep1\\rep1_image009-1.tif',
 'CEN7\\Cen7_AuroraB_CENPA\\crop_rep1\\rep1_image010-1.tif',
 'CEN7\\Cen7_AuroraB_CENPA\\crop_rep1\\rep1_image013-1.tif',
 'CEN7\\Cen7_AuroraB_CENPA\\crop_rep1\\rep1_image013-2.tif',
 'CEN7\\Cen7_AuroraB_CENPA\\crop_rep1\\rep1_image015-1.tif',
 'CEN7\\Cen7_AuroraB_CEN

In [12]:
# View example to make sure everything works
process_file('CEN7/Cen7_CENPB_CENPA/crop_rep2/image008-1.tif', viewer=viewer, display=True)

(17, 4, 484, 554)
Bright peak distance 4.367743394488857 or dim peak distance 96.74877068298096 exceeds max distance 20.


""


In [19]:
df = pd.DataFrame()
for fname in fnames:
    print(fname)
    if 'SUM_' in fname:
        print("Skipping SUM file")
        continue
    prop_df = process_file(fname, viewer=viewer, display=False, max_distance=20)
    prop_df['filename'] = fname
    df = pd.concat([df, prop_df])


CEN7\Cen7_AuroraB_CENPA\crop_rep1\rep1_image001-1.tif
(33, 4, 353, 191)
CEN7\Cen7_AuroraB_CENPA\crop_rep1\rep1_image003-1.tif
(37, 4, 252, 243)
CEN7\Cen7_AuroraB_CENPA\crop_rep1\rep1_image003-2.tif
(37, 4, 208, 185)
CEN7\Cen7_AuroraB_CENPA\crop_rep1\rep1_image004-1.tif
(31, 4, 348, 412)
CEN7\Cen7_AuroraB_CENPA\crop_rep1\rep1_image005-1.tif
(31, 4, 481, 316)
CEN7\Cen7_AuroraB_CENPA\crop_rep1\rep1_image005-2.tif
(31, 4, 331, 279)
Bright peak distance 21.457276250051272 or dim peak distance 2.3038664139907707 exceeds max distance 20.
CEN7\Cen7_AuroraB_CENPA\crop_rep1\rep1_image005-3.tif
(31, 4, 268, 320)
CEN7\Cen7_AuroraB_CENPA\crop_rep1\rep1_image006-1.tif
(34, 4, 487, 261)
CEN7\Cen7_AuroraB_CENPA\crop_rep1\rep1_image006-2.tif
(34, 4, 126, 247)
Bright peak distance 6.029776172632075 or dim peak distance 29.07227064170779 exceeds max distance 20.
CEN7\Cen7_AuroraB_CENPA\crop_rep1\rep1_image007-1.tif
(29, 4, 312, 236)
CEN7\Cen7_AuroraB_CENPA\crop_rep1\rep1_image008-1.tif
(35, 4, 236, 222)


# Analyze Data

In [20]:
df['State'] = 'Bright'
df.iloc[1::2, df.columns.get_loc('State')] = 'Dim'

In [21]:
df['folder'] = df['filename'].str.split('\\').str[1]
df['rep'] = df['filename'].str.split('\\').str[2]
df.groupby(['folder', 'rep']).agg({'label': 'count','slices':np.mean}).reset_index()

C:\Users\smc\AppData\Local\Temp\ipykernel_224\145448295.py:3: FutureWarning: The provided callable <function mean at 0x00000184FF1825C0> is currently using SeriesGroupBy.mean. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "mean" instead.
  df.groupby(['folder', 'rep']).agg({'label': 'count','slices':np.mean}).reset_index()


,folder,rep,label,slices
0,Cen7_AuroraB_CENPA,crop_rep1,28,34.714286
1,Cen7_AuroraB_CENPA,crop_rep2,34,30.705882
2,Cen7_AuroraB_CENPA,crop_rep3_left,40,40.200000
3,Cen7_CENPB_CENPA,crop_rep0,16,11.250000
4,Cen7_CENPB_CENPA,crop_rep1,40,19.500000
5,Cen7_CENPB_CENPA,crop_rep2,18,21.111111
6,Cen7_CENPB_CENPA,crop_rep3,20,14.800000
7,Cen7_CENPE_CENPA,crop_rep1,28,35.928571
8,Cen7_CENPE_CENPA,crop_rep2,28,41.000000
9,Cen7_CENPE_CENPA,crop_rep3,44,29.045455


In [22]:
bright_df = df[df['State'] == 'Bright']
dim_df = df[df['State'] == 'Dim']

combined_df = bright_df.merge(dim_df, on=['filename', 'folder', ], suffixes=('_bright', '_dim'))
combined_df['FISH_ratio']= combined_df['FISH_total_intensity_bright'] / combined_df['FISH_total_intensity_dim']
combined_df['c0_ratio'] = combined_df['total_intensity_bright'] / combined_df['total_intensity_dim']
combined_df['c1_ratio'] = combined_df['total_intensity_c1_bright'] / combined_df['total_intensity_c1_dim']
combined_df['c2_ratio'] = combined_df['total_intensity_c2_bright'] / combined_df['total_intensity_c2_dim']
combined_df['c3_ratio'] = combined_df['total_intensity_c3_bright'] / combined_df['total_intensity_c3_dim']

combined_df['FISH_ratio_i'] = 1 / combined_df['FISH_ratio']
combined_df['c0_ratio_i'] = 1 / combined_df['c0_ratio']
combined_df['c1_ratio_i'] = 1 / combined_df['c1_ratio']
combined_df['c2_ratio_i'] = 1 / combined_df['c2_ratio']
combined_df['c3_ratio_i'] = 1 / combined_df['c3_ratio']
combined_df

,label_bright,centroid-0_bright,centroid-1_bright,area_bright,mean_intensity_bright,total_intensity_bright,mean_intensity_c1_bright,total_intensity_c1_bright,mean_intensity_c2_bright,total_intensity_c2_bright,...,FISH_ratio,c0_ratio,c1_ratio,c2_ratio,c3_ratio,FISH_ratio_i,c0_ratio_i,c1_ratio_i,c2_ratio_i,c3_ratio_i
0,1.0,324.0,89.5,502.0,511.617523,256831.996643,480.519928,241221.003845,5957.073730,2.990451e+06,...,1.499137,1.422080,1.239204,1.186483,1.454172,0.667051,0.703195,0.806970,0.842827,0.687677
1,1.0,75.0,158.5,484.0,1557.593018,753875.020508,789.109497,381928.996582,21882.462891,1.059111e+07,...,2.359160,1.537482,1.549879,2.243623,2.057317,0.423880,0.650414,0.645212,0.445708,0.486070
2,1.0,59.5,58.0,436.0,2013.575684,877918.998047,786.371582,342858.009766,7039.536621,3.069238e+06,...,2.792112,2.057211,1.262564,1.181580,0.994089,0.358152,0.486095,0.792039,0.846325,1.005947
3,1.0,66.0,87.5,426.0,1720.739380,733034.975830,710.589172,302710.987427,13419.406250,5.716667e+06,...,2.359123,1.836177,1.017588,0.768440,1.531633,0.423886,0.544610,0.982716,1.301338,0.652898
4,1.0,398.0,100.0,355.0,1443.949341,512602.015991,492.380280,174794.999237,4190.126953,1.487495e+06,...,2.371269,1.659180,0.701672,1.046452,1.378087,0.421715,0.602707,1.425167,0.955610,0.725644
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
347,1.0,942.5,262.0,506.0,1836.120605,929077.026367,486.393280,246114.999695,4108.541504,2.078922e+06,...,1.585129,1.439722,0.867411,0.706706,0.499809,0.630863,0.694579,1.152855,1.415017,2.000765
348,1.0,369.0,309.0,517.0,1146.889771,592942.011353,645.079285,333505.990173,5935.404297,3.068604e+06,...,1.605485,1.253275,1.067025,0.738341,0.537941,0.622865,0.797909,0.937186,1.354388,1.858941
349,1.0,162.5,297.0,326.0,566.717773,184749.994141,731.404907,238437.999756,5833.717773,1.901792e+06,...,1.587022,0.733298,1.109633,2.072186,1.366198,0.630111,1.363702,0.901199,0.482582,0.731959
350,1.0,213.0,302.0,469.0,1688.545898,791928.026367,536.729187,251725.988708,4483.110840,2.102579e+06,...,2.596470,2.234062,0.852294,1.086778,0.976730,0.385138,0.447615,1.173304,0.920151,1.023825


In [23]:
# Make interactive plot for examining FISH signal ratios

f=go.FigureWidget(
    #px.box(df, x='folder', color='State', y='total_intensity_c3', points='all', hover_data=['filename'])
    px.box(combined_df, x='folder', y='FISH_ratio_i', points='all', hover_data=['filename'], range_y=[0,2], width=1400, height=800)
    )

def click_fn(trace, points, state):
    
    if (len(points.point_inds)>0):
        idx = f.data[points.trace_index]['customdata'][points.point_inds[-1]][0]
        print(idx)
        viewer.layers.clear()
        process_file(idx, viewer=viewer, display=True)

for a in f.data:
    a.on_click(click_fn)
f.write_html('FISH_signal_ratio.html')
f

FigureWidget({
    'data': [{'alignmentgroup': 'True',
              'boxpoints': 'all',
              'customdata': array([['CEN7\\Cen7_AuroraB_CENPA\\crop_rep1\\rep1_image001-1.tif'],
                                   ['CEN7\\Cen7_AuroraB_CENPA\\crop_rep1\\rep1_image003-1.tif'],
                                   ['CEN7\\Cen7_AuroraB_CENPA\\crop_rep1\\rep1_image003-2.tif'],
                                   ...,
                                   ['CEN7\\Cen7_TopoIIA_CENPA\\crop_rep3\\Image015-1.tif'],
                                   ['CEN7\\Cen7_TopoIIA_CENPA\\crop_rep3\\Image016-1.tif'],
                                   ['CEN7\\Cen7_TopoIIA_CENPA\\crop_rep3\\Image017-1.tif']], dtype=object),
              'hovertemplate': 'folder=%{x}<br>FISH_ratio_i=%{y}<br>filename=%{customdata[0]}<extra></extra>',
              'legendgroup': '',
              'marker': {'color': '#636efa'},
              'name': '',
              'notched': False,
              'offsetgroup': '',
       

In [24]:
# Make interactive plot for examining CenpA signal ratios

f=go.FigureWidget(
    #px.box(df, x='folder', color='State', y='total_intensity_c3', points='all', hover_data=['filename'])
    px.box(combined_df, x='folder', y='c1_ratio_i', points='all', hover_data=['filename'], range_y=[0,2])
    )

def click_fn(trace, points, state):
    
    if (len(points.point_inds)>0):
        idx = f.data[points.trace_index]['customdata'][points.point_inds[-1]][0]
        print(idx)
        viewer.layers.clear()
        process_file(idx, viewer=viewer, display=True)

for a in f.data:
    a.on_click(click_fn)
f.write_html('cenpA_signal_ratio.html')
f

FigureWidget({
    'data': [{'alignmentgroup': 'True',
              'boxpoints': 'all',
              'customdata': array([['CEN7\\Cen7_AuroraB_CENPA\\crop_rep1\\rep1_image001-1.tif'],
                                   ['CEN7\\Cen7_AuroraB_CENPA\\crop_rep1\\rep1_image003-1.tif'],
                                   ['CEN7\\Cen7_AuroraB_CENPA\\crop_rep1\\rep1_image003-2.tif'],
                                   ...,
                                   ['CEN7\\Cen7_TopoIIA_CENPA\\crop_rep3\\Image015-1.tif'],
                                   ['CEN7\\Cen7_TopoIIA_CENPA\\crop_rep3\\Image016-1.tif'],
                                   ['CEN7\\Cen7_TopoIIA_CENPA\\crop_rep3\\Image017-1.tif']], dtype=object),
              'hovertemplate': 'folder=%{x}<br>c1_ratio_i=%{y}<br>filename=%{customdata[0]}<extra></extra>',
              'legendgroup': '',
              'marker': {'color': '#636efa'},
              'name': '',
              'notched': False,
              'offsetgroup': '',
         

In [25]:
# Make interactive plot for examining kinteochore protein signal ratios
f=go.FigureWidget(
    px.box(combined_df, x='folder', y='c2_ratio_i', points='all', hover_data=['filename'], range_y=[0,2], height=800)
    )

def click_fn(trace, points, state):
    
    if (len(points.point_inds)>0):
        idx = f.data[points.trace_index]['customdata'][points.point_inds[-1]][0]
        print(idx)
        process_file(idx, viewer=viewer, display=True)
        for layer in viewer.layers:
            layer.visible = False
        viewer.layers[-1].visible = True
        viewer.layers[-4].visible = True
        viewer.layers[-5].visible = True
        viewer.layers[-6].visible = True
        viewer.layers[-1].name = idx

for a in f.data:
    a.on_click(click_fn)
f.write_html('core_protein_signal_ratio.html')
f

FigureWidget({
    'data': [{'alignmentgroup': 'True',
              'boxpoints': 'all',
              'customdata': array([['CEN7\\Cen7_AuroraB_CENPA\\crop_rep1\\rep1_image001-1.tif'],
                                   ['CEN7\\Cen7_AuroraB_CENPA\\crop_rep1\\rep1_image003-1.tif'],
                                   ['CEN7\\Cen7_AuroraB_CENPA\\crop_rep1\\rep1_image003-2.tif'],
                                   ...,
                                   ['CEN7\\Cen7_TopoIIA_CENPA\\crop_rep3\\Image015-1.tif'],
                                   ['CEN7\\Cen7_TopoIIA_CENPA\\crop_rep3\\Image016-1.tif'],
                                   ['CEN7\\Cen7_TopoIIA_CENPA\\crop_rep3\\Image017-1.tif']], dtype=object),
              'hovertemplate': 'folder=%{x}<br>c2_ratio_i=%{y}<br>filename=%{customdata[0]}<extra></extra>',
              'legendgroup': '',
              'marker': {'color': '#636efa'},
              'name': '',
              'notched': False,
              'offsetgroup': '',
         

In [26]:
df.groupby(['folder']).agg({'label':'size', 'filename':lambda x: len(np.unique(x))}).reset_index()

,folder,label,filename
0,Cen7_AuroraB_CENPA,102,51
1,Cen7_CENPB_CENPA,94,47
2,Cen7_CENPE_CENPA,100,50
3,Cen7_INCENP_CENPA,96,48
4,Cen7_KNL1_CENPA,96,48
5,Cen7_NDC80_CENPA,60,30
6,Cen7_SGO_CENPA,54,27
7,Cen7_TopoIIA_CENPA,102,51


In [27]:
combined_df.to_csv('combined_peak_datav3.csv')